In [ ]:
// Function to get student email and name based on roll numberfunction GetStudentEmail() {  var ss = SpreadsheetApp.getActiveSpreadsheet();  var minicexSheet = ss.getSheetByName("MiniCEX");  var residentSheet = ss.getSheetByName("Residents");  var lastRow = minicexSheet.getLastRow();  var rollNumber = minicexSheet.getRange(lastRow, 3).getValue();  var residentData = residentSheet.getDataRange().getValues();  for (var i = 1; i < residentData.length; i++) {    if (residentData[i][0] == rollNumber) {      var studentName = residentData[i][1];      var studentEmail = residentData[i][2];      return { studentName: studentName, studentEmail: studentEmail };    }  }  return { studentName: "", studentEmail: "" };}// Function to calculate total marks, full marks, and percentagefunction calculate() {  var ss = SpreadsheetApp.getActiveSpreadsheet();  var minicexSheet = ss.getSheetByName("MiniCEX");  var lastRow = minicexSheet.getLastRow();  var data = minicexSheet.getRange(lastRow, 10, 1, 6).getValues()[0];  var totalMark = 0;  var zeroCount = 0;  for (var i = 0; i < data.length; i++) {    var val = Number(data[i]);    if (val === 0) zeroCount++;    totalMark += val;  }  var fullMark = 54 - (zeroCount * 9);  var percentage = (totalMark / fullMark) * 100;  return { totalMark: totalMark, fullMark: fullMark, percentage: percentage };}// Function to send email to studentfunction SendEmailStudent() {  var ss = SpreadsheetApp.getActiveSpreadsheet();  var minicexSheet = ss.getSheetByName("MiniCEX");  var lastRow = minicexSheet.getLastRow();  var studentInfo = GetStudentEmail();  var calc = calculate();  var values = minicexSheet.getRange(lastRow, 2, 1, 20).getValues()[0];  var body = `Evaluating Faculty: ${values[0]}Position of Faculty: ${values[19]}Resident Name: ${studentInfo.studentName}Resident Roll Number: ${values[1]}Year: ${values[2]}Date of Assessment: ${values[3]}Speciality: ${values[4]}EPA Topic: ${values[5]}EPA Number: ${values[6]}Clinical Case: ${values[7]}Medical Interviewing: ${values[8]}Physical Examination: ${values[9]}Professionalism and Rapport: ${values[10]}Clinical Reasoning and Judgment: ${values[11]}Organization and Efficiency: ${values[12]}Overall Clinical Competence: ${values[13]}Obtained Marks: ${calc.totalMark}Obtained Percentage: ${calc.percentage.toFixed(2)}%Overall ability to perform procedure: ${values[14]}Areas done well: ${values[15]}Specific action that needs to be taken to achieve desired competency: ${values[16]}Thank youDean Office`;  var subject = `MiniCEX; Year ${values[2]}, EPA ${values[5]}, EpA no ${values[6]}`;  MailApp.sendEmail(studentInfo.studentEmail, subject, body);}// Function to get Program Director's name and emailfunction ProgramDirector() {  var ss = SpreadsheetApp.getActiveSpreadsheet();  var minicexSheet = ss.getSheetByName("MiniCEX");  var pdSheet = ss.getSheetByName("ProgramDirector");  var lastRow = minicexSheet.getLastRow();  var speciality = minicexSheet.getRange(lastRow, 6).getValue();  var pdData = pdSheet.getDataRange().getValues();  for (var i = 1; i < pdData.length; i++) {    if (pdData[i][0] == speciality) {      return { programDirectorName: pdData[i][1], programDirectorEmail: pdData[i][2] };    }  }  return { programDirectorName: "", programDirectorEmail: "" };}// Function to assess faculty performancefunction FacultyPerformance() {  var ss = SpreadsheetApp.getActiveSpreadsheet();  var minicexSheet = ss.getSheetByName("MiniCEX");  var lastRow = minicexSheet.getLastRow();  var data = minicexSheet.getRange(lastRow, 10, 1, 6).getValues()[0];  var calc = calculate();  var mean = data.reduce((a, b) => a + b, 0) / data.length;  var varianceSum = data.reduce((a, b) => a + Math.pow(b - mean, 2), 0);  var stanDev = Math.sqrt(varianceSum / data.length);  var variance = stanDev / mean;  var discrimination = variance === 0 ? "Poor" : (variance < 0.5 ? "Good" : "Moderate");  var areaWell = minicexSheet.getRange(lastRow, 17).getValue();  var areaImp = minicexSheet.getRange(lastRow, 18).getValue();  var cleanText = text => text.replace(/[.,\/#!$%\^&\*;:{}=\-_`~()]/g, '').split(/\s+/).filter(w => w.length >= 4);  var lenAreaImp = cleanText(areaImp).length;  var narrativeSkill = lenAreaImp <= 5 ? "Needs Improvement" : (lenAreaImp <= 10 ? "Can be better" : "Is Great");  return {    mean: mean,    stanDev: stanDev,    variance: variance,    discrimination: discrimination,    narrativeSkill: narrativeSkill  };}// Function to place calculated values in the last rowfunction Place() {  var ss = SpreadsheetApp.getActiveSpreadsheet();  var minicexSheet = ss.getSheetByName("MiniCEX");  var lastRow = minicexSheet.getLastRow();  var calc = calculate();  var perf = FacultyPerformance();  minicexSheet.getRange(lastRow, 26).setValue(calc.fullMark);  minicexSheet.getRange(lastRow, 27).setValue(calc.totalMark);  minicexSheet.getRange(lastRow, 28).setValue(calc.percentage);  minicexSheet.getRange(lastRow, 29).setValue(perf.mean);  minicexSheet.getRange(lastRow, 30).setValue(perf.stanDev);  minicexSheet.getRange(lastRow, 31).setValue(perf.variance);  minicexSheet.getRange(lastRow, 32).setValue(perf.discrimination);  minicexSheet.getRange(lastRow, 33).setValue(perf.narrativeSkill);}// Function to send Program Director email (and also to MiniCEX sheet column B)function SendEmailProgramDirector() {  var ss = SpreadsheetApp.getActiveSpreadsheet();  var minicexSheet = ss.getSheetByName("MiniCEX");  var lastRow = minicexSheet.getLastRow();  var studentInfo = GetStudentEmail();  var calc = calculate();  var pd = ProgramDirector();  var perf = FacultyPerformance();  var values = minicexSheet.getRange(lastRow, 2, 1, 20).getValues()[0];  var ccEmail = minicexSheet.getRange(lastRow, 2).getValue();  // Column B  var body = `Program Director: ${pd.programDirectorName}Evaluating Faculty: ${values[0]}Position of Faculty: ${values[19]}Resident Name: ${studentInfo.studentName}Resident Roll Number: ${values[1]}Year: ${values[2]}Speciality: ${values[4]}EPA Topic: ${values[5]}EPA Number: ${values[6]}Student EvaluationClinical Case: ${values[7]}Medical Interviewing: ${values[8]}Physical Examination: ${values[9]}Professionalism and Rapport: ${values[10]}Clinical Reasoning and Judgment: ${values[11]}Organization and Efficiency: ${values[12]}Overall Clinical Competence: ${values[13]}Obtained Marks: ${calc.totalMark}Obtained Percentage: ${calc.percentage.toFixed(2)}%Overall ability to perform procedure: ${values[14]}Areas done well: ${values[15]}Specific action that needs to be taken to achieve desired competency: ${values[16]}Faculty Performance(This is based on scoring pattern and may not reflect truly, however it is helpful for program director to identify extreme pattern repeatedly.)The ability of faculty for identifying skills across various items seems to be ${perf.discrimination}.The skill of faculty in narrative writeup: ${perf.narrativeSkill}.Thank youDean Office`;  var subject = `MiniCEX; Year ${values[2]}, EPA ${values[5]}, EpA no ${values[6]}`;  MailApp.sendEmail({    to: pd.programDirectorEmail,    cc: ccEmail,    subject: subject,    body: body  });}